In [ ]:
# Importation des librairies nécessaires
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold, learning_curve
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.metrics import classification_report, roc_curve, auc, accuracy_score, precision_recall_curve
from sklearn.metrics import confusion_matrix, average_precision_score, f1_score
from sklearn.feature_selection import RFECV, mutual_info_classif

from sklearn.ensemble import IsolationForest
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.pipeline import Pipeline as ImbPipeline

# Set display precision & style
np.set_printoptions(precision=4)
pd.set_option('display.precision', 4)
sns.set_style('whitegrid')
plt.style.use('seaborn-v0_8-pastel')

# ---------------------------
# 0. Chargement et exploration du dataset
# ---------------------------
print("="*80)
print("PHISHING DETECTION AVEC LIGHTGBM - ANALYSE AVANCÉE")
print("="*80)

# Charger le dataset
df = pd.read_csv("Phishing_Legitimate_full.csv")

print(f"Dimensions du dataset: {df.shape[0]} lignes × {df.shape[1]} colonnes")

# Créer une figure avec deux subplots pour l'exploration initiale
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# Afficher la distribution des classes
target_column = 'CLASS_LABEL' if 'CLASS_LABEL' in df.columns else 'label'
class_counts = df[target_column].value_counts()
colors = ['#6495ED', '#FF7F50']
ax1.pie(class_counts, labels=class_counts.index, autopct='%1.1f%%',
        colors=colors, shadow=True, startangle=90, explode=(0.1, 0))
ax1.set_title('Distribution des Classes', fontsize=14, pad=20)

# Afficher les statistiques descriptives pour quelques colonnes numériques
numeric_columns = df.select_dtypes(include=['int64', 'float64']).columns[:10]
stats_df = df[numeric_columns].describe().T[['mean', 'std', 'min', 'max']]
sns.heatmap(stats_df, cmap='coolwarm', annot=True, fmt='.2f', ax=ax2)
ax2.set_title('Statistiques Descriptives (10 premières variables)', fontsize=14, pad=20)

plt.tight_layout()
plt.show()

# ---------------------------
# 1. Préparation avancée des données
# ---------------------------
print("\n" + "="*80)
print("PRÉPARATION ET PRÉTRAITEMENT DES DONNÉES")
print("="*80)

# Retirer les colonnes non pertinentes (exemple : 'id' et 'url')
for col in ['id', 'url']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)

# Vérification et traitement des valeurs manquantes
missing_values = df.isnull().sum()
print("\nValeurs manquantes par colonne:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "Aucune valeur manquante")

# Suppression des observations avec une valeur manquante dans la target (si nécessaire)
df.dropna(subset=[target_column], inplace=True)

# Vérifier les corrélations entre les variables
corr_matrix = df.corr()
plt.figure(figsize=(16, 12))
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask)] = True
cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(corr_matrix, mask=mask, cmap=cmap, vmax=1, vmin=-1, center=0,
            square=True, linewidths=.5, annot=False, cbar_kws={'shrink': .5})
plt.title('Matrice de Corrélation des Variables', fontsize=16, pad=20)
plt.tight_layout()
plt.show()

# Séparation des features et de la cible
X = df.drop(columns=[target_column])
y = df[target_column]
feature_names = X.columns

# Analyse des caractéristiques initiales
print(f"\nNombre de caractéristiques initiales: {X.shape[1]}")
print(f"Distribution initiale des classes: {y.value_counts().to_dict()}")

# ---------------------------
# 2. Prétraitement avancé: Scaling avec PowerTransformer + StandardScaler
# ---------------------------
# PowerTransformer pour normaliser les distributions
print("\nApplication de transformations pour normaliser les distributions...")
power_transformer = PowerTransformer(method='yeo-johnson')
X_transformed = power_transformer.fit_transform(X)

# Standard Scaler pour centrer et réduire
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_transformed)
print("Transformations appliquées.")

# ---------------------------
# 3. Détection et suppression des anomalies avec IsolationForest
# ---------------------------
print("\nDétection d'anomalies avec IsolationForest:")
chosen_contamination = 0.01
iso = IsolationForest(contamination=chosen_contamination, random_state=42, n_jobs=-1)
pred_anomalies = iso.fit_predict(X_scaled)
mask = pred_anomalies != -1
X_clean = X_scaled[mask]
y_clean = y[mask]

# Visualisation des anomalies (PCA pour la visualisation en 2D)
from sklearn.decomposition import PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
plt.scatter(X_pca[mask, 0], X_pca[mask, 1], c=y[mask], cmap='viridis', alpha=0.6, s=30, edgecolor='k', linewidth=0.3)
plt.scatter(X_pca[~mask, 0], X_pca[~mask, 1], c='red', marker='x', s=50, label='Anomalies')
plt.title('Visualisation des Anomalies Détectées (PCA)', fontsize=14)
plt.xlabel('Composante Principale 1')
plt.ylabel('Composante Principale 2')
plt.legend()
plt.tight_layout()
plt.show()

nombre_anomalies = len(X_scaled) - np.sum(mask)
print(f"Points supprimés comme anomalies: {nombre_anomalies} ({nombre_anomalies/len(X_scaled)*100:.2f}%)")

# ---------------------------
# 4. Équilibrage de la dataset avec SMOTE
# ---------------------------
print("\nÉquilibrage des classes avec SMOTE...")
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X_clean, y_clean)

# Visualisation avant/après équilibrage
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Avant SMOTE
ax1.bar(['Légitime', 'Phishing'], [np.sum(y_clean == 0), np.sum(y_clean == 1)],
        color=['#6495ED', '#FF7F50'])
ax1.set_title('Distribution des Classes - Avant SMOTE', fontsize=14)
ax1.set_ylabel('Nombre d\'échantillons')

# Après SMOTE
ax2.bar(['Légitime', 'Phishing'], [np.sum(y_res == 0), np.sum(y_res == 1)],
        color=['#6495ED', '#FF7F50'])
ax2.set_title('Distribution des Classes - Après SMOTE', fontsize=14)
ax2.set_ylabel('Nombre d\'échantillons')

plt.tight_layout()
plt.show()

print(f"Distribution des classes après SMOTE: {np.bincount(y_res)}")

# ---------------------------
# 5. RFECV pour la sélection optimale des features
# ---------------------------
print("\n" + "="*80)
print("SÉLECTION DE CARACTÉRISTIQUES AVEC RFECV")
print("="*80)

# Utiliser un modèle LGBM comme base pour RFECV
base_lgbm = lgb.LGBMClassifier(random_state=42, n_jobs=-1)

# Configuration RFECV pour sélection optimale des features
rfecv = RFECV(
    estimator=base_lgbm,
    step=1,  # Éliminer 1 feature à chaque étape
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1',  # F1 score comme métrique (bien adapté aux problèmes déséquilibrés)
    n_jobs=-1,  # Utiliser tous les processeurs disponibles
    verbose=1,
    min_features_to_select=5  # Garder au moins 5 features
)

print("\nExécution de la sélection de features avec RFECV (patience, cela peut prendre du temps)...")
rfecv.fit(X_res, y_res)

# Résultats RFECV
print(f"\nNombre optimal de features: {rfecv.n_features_}")
print(f"Score avec {rfecv.n_features_} features: {rfecv.cv_results_['mean_test_score'][rfecv.n_features_ - 1]:.4f}")

# Visualisation des scores RFECV
plt.figure(figsize=(12, 6))
plt.plot(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'], marker='o',
         color='#6495ED', linestyle='-', linewidth=2, markersize=8)
plt.axvline(x=rfecv.n_features_, color='#FF7F50', linestyle='--', label=f'Optimal: {rfecv.n_features_} features')
plt.fill_between(range(1, len(rfecv.cv_results_['mean_test_score']) + 1), rfecv.cv_results_['mean_test_score'] - np.std(rfecv.cv_results_['mean_test_score']),
                rfecv.cv_results_['mean_test_score'] + np.std(rfecv.cv_results_['mean_test_score']), alpha=0.2, color='#6495ED')
plt.xlabel('Nombre de Features', fontsize=12)
plt.ylabel('Score de Validation Croisée (F1)', fontsize=12)
plt.title('Scores de Validation Croisée vs Nombre de Features', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()
plt.tight_layout()
plt.show()

# Récupérer les features sélectionnées
selected_features_mask = rfecv.support_
selected_feature_names = feature_names[selected_features_mask]

# Afficher les features sélectionnées
print("\nFeatures sélectionnées par RFECV:")
for i, feature in enumerate(selected_feature_names):
    print(f"{i+1}. {feature}")

# Transformer les données pour garder uniquement les features sélectionnées
X_selected = rfecv.transform(X_res)

# ---------------------------
# 6. Analyse d'importance des features
# ---------------------------
# Calculer l'importance des features via information mutuelle
mi_scores = mutual_info_classif(X_selected, y_res, random_state=42)
mi_df = pd.DataFrame({'Feature': selected_feature_names, 'Importance': mi_scores})
mi_df = mi_df.sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=mi_df, palette='viridis')
plt.title('Importance des Features (Information Mutuelle)', fontsize=14)
plt.tight_layout()
plt.show()

# ---------------------------
# 7. Split train/test
# ---------------------------
print("\n" + "="*80)
print("ENTRAINEMENT ET OPTIMISATION DU MODÈLE LGBM")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_selected, y_res, test_size=0.2, random_state=42, stratify=y_res
)

print(f"Dimensions de l'ensemble d'entraînement: {X_train.shape}")
print(f"Dimensions de l'ensemble de test: {X_test.shape}")

# ---------------------------
# 8. Recherche d'hyperparamètres pour LGBM
# ---------------------------
print("\nRecherche des hyperparamètres optimaux pour LightGBM...")

# Préparation des datasets LGBM
lgb_train = lgb.Dataset(X_train, y_train)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train)

# Paramètres de base pour LightGBM
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0,
    'num_threads': -1,
    'lambda_l1': 0.1,
    'lambda_l2': 0.1
}

# Grid Search CV pour les hyperparamètres
param_grid = {
    'num_leaves': [31, 50, 75],
    'learning_rate': [0.01, 0.05, 0.1],
    'n_estimators': [100, 200, 300],
    'reg_alpha': [0, 0.1, 0.5],
    'reg_lambda': [0, 0.1, 0.5],
    'min_child_samples': [20, 50, 100]
}

lgbm_model = lgb.LGBMClassifier(random_state=42, objective='binary')
grid_search = GridSearchCV(
    lgbm_model, param_grid,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring='f1', n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

# Afficher les meilleurs paramètres
print(f"\nMeilleurs paramètres trouvés: {grid_search.best_params_}")
print(f"Meilleur score de validation: {grid_search.best_score_:.4f}")

# ---------------------------
# 9. Entraînement approfondi du modèle LGBM
# ---------------------------
# Utiliser les meilleurs paramètres trouvés
best_params = grid_search.best_params_
best_params['objective'] = 'binary'
best_params['metric'] = ['binary_logloss', 'auc']
best_params['verbose'] = -1

print("\nEntraînement approfondi du modèle LightGBM avec les meilleurs paramètres...")
callbacks = [
    lgb.early_stopping(stopping_rounds=50, verbose=True),
    lgb.log_evaluation(period=20)
]

lgbm_model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_eval],
    callbacks=callbacks
)

print(f"\nMeilleure itération: {lgbm_model.best_iteration}")

# ---------------------------
# 10. Évaluation du modèle
# ---------------------------
print("\n" + "="*80)
print("ÉVALUATION ET VISUALISATION DES PERFORMANCES")
print("="*80)

# Prédictions
y_prob = lgbm_model.predict(X_test)
y_pred = (y_prob >= 0.5).astype(int)

# Classification report
print("\n--- Rapport de Classification ---")
report = classification_report(y_test, y_pred, digits=4)
print(report)

# Visualisation de la courbe d'apprentissage
def plot_learning_curve(estimator, X, y, axes=None, ylim=None, cv=None,
                        n_jobs=None, train_sizes=np.linspace(.1, 1.0, 10)):
    """Generate & plot learning curve: performance as function of training size."""

    train_sizes, train_scores, test_scores = learning_curve(
        estimator, X, y, cv=cv, n_jobs=n_jobs, train_sizes=train_sizes)

    train_scores_mean = np.mean(train_scores, axis=1)
    train_scores_std = np.std(train_scores, axis=1)
    test_scores_mean = np.mean(test_scores, axis=1)
    test_scores_std = np.std(test_scores, axis=1)

    # Plot learning curve
    plt.figure(figsize=(10, 6))
    plt.grid(True, alpha=0.3)

    plt.fill_between(train_sizes, train_scores_mean - train_scores_std,
                     train_scores_mean + train_scores_std, alpha=0.1,
                     color="r")
    plt.fill_between(train_sizes, test_scores_mean - test_scores_std,
                     test_scores_mean + test_scores_std, alpha=0.1,
                     color="g")
    plt.plot(train_sizes, train_scores_mean, 'o-', color="r",
             label="Score d'entraînement")
    plt.plot(train_sizes, test_scores_mean, 'o-', color="g",
             label="Score de validation croisée")

    plt.title("Courbe d'Apprentissage LightGBM", fontsize=14)
    plt.xlabel("Taille de l'ensemble d'entraînement", fontsize=12)
    plt.ylabel("Score (Accuracy)", fontsize=12)
    plt.legend(loc="best")

    plt.tight_layout()
    return plt

# Générer la courbe d'apprentissage
lgbm_clf = lgb.LGBMClassifier(**{k: v for k, v in best_params.items()
                                if k not in ['metric', 'verbose']})
plot_learning_curve(
    lgbm_clf, X_selected, y_res,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1
)
plt.show()

# Afficher la matrice de confusion
plt.figure(figsize=(8, 7))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Matrice de Confusion', fontsize=14)
plt.ylabel('Valeur Réelle', fontsize=12)
plt.xlabel('Valeur Prédite', fontsize=12)
plt.tight_layout()
plt.show()

# ROC et PR curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
ax1.plot(fpr, tpr, color='darkorange', lw=2, label=f'AUC = {roc_auc:.4f}')
ax1.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax1.set_xlim([0.0, 1.0])
ax1.set_ylim([0.0, 1.05])
ax1.set_xlabel('Taux de Faux Positifs', fontsize=12)
ax1.set_ylabel('Taux de Vrais Positifs', fontsize=12)
ax1.set_title('Courbe ROC', fontsize=14)
ax1.legend(loc="lower right")
ax1.grid(True, alpha=0.3)

# Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)
ax2.plot(recall, precision, color='blue', lw=2, label=f'AP = {pr_auc:.4f}')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('Recall', fontsize=12)
ax2.set_ylabel('Precision', fontsize=12)
ax2.set_title('Courbe Precision-Recall', fontsize=14)
ax2.legend(loc="lower left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ---------------------------
# 11. Feature importance
# ---------------------------
# Visualisation de l'importance des features
plt.figure(figsize=(12, 8))
lgb.plot_importance(lgbm_model, max_num_features=20, figsize=(12, 8), title="Importance des Features (LightGBM)")
plt.tight_layout()
plt.show()

# ---------------------------
# 12. Analyse d'erreurs
# ---------------------------
# Indices où les prédictions sont fausses
error_indices = np.where(y_pred != y_test)[0]

if len(error_indices) > 0:
    print(f"\nAnalyse des {min(10, len(error_indices))} premières erreurs:")

    for i, idx in enumerate(error_indices[:10]):
        print(f"Exemple {i+1}:")
        # Convertir y_test et y_pred en array numpy si ce n'est pas déjà fait
        y_test_array = np.array(y_test) if not isinstance(y_test, np.ndarray) else y_test
        y_pred_array = np.array(y_pred) if not isinstance(y_pred, np.ndarray) else y_pred

        print(f"  Prédiction: {y_pred_array[idx]}, Vrai label: {y_test_array[idx]}")
        print(f"  Probabilité prédite: {y_prob[idx]:.4f}")
        print()

# ---------------------------
# 13. Analyse 3D des features importantes
# ---------------------------
# Sélectionner les 3 features les plus importantes pour la visualisation 3D
top_features_idx = np.argsort(lgbm_model.feature_importance())[-3:]
X_top3 = X_test[:, top_features_idx]

# Créer un scatter plot 3D
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')

# Colorer par classe réelle
scatter = ax.scatter(X_top3[:, 0], X_top3[:, 1], X_top3[:, 2],
                     c=y_test, cmap='coolwarm', s=50, alpha=0.6, edgecolor='k', linewidth=0.3)

# Marquer les erreurs de prédiction
ax.scatter(X_top3[error_indices, 0], X_top3[error_indices, 1], X_top3[error_indices, 2],
           marker='x', s=100, color='black', label='Erreurs')

# Ajouter des titres
ax.set_title('Visualisation 3D des 3 Features les Plus Importantes', fontsize=14)
ax.set_xlabel(f'Feature {top_features_idx[0]}', fontsize=12)
ax.set_ylabel(f'Feature {top_features_idx[1]}', fontsize=12)
ax.set_zlabel(f'Feature {top_features_idx[2]}', fontsize=12)

# Ajouter une légende pour les classes
legend1 = ax.legend(*scatter.legend_elements(), title="Classes")
ax.add_artist(legend1)
ax.legend()

# Améliorer la présentation
ax.view_init(30, 45)  # Définir l'angle de vue
plt.tight_layout()
plt.show()

# ---------------------------
# 14. Sauvegarde du modèle
# ---------------------------
print("\nSauvegarde du modèle LightGBM...")
lgbm_model.save_model('lightgbm_phishing_model.txt')
print("Modèle sauvegardé sous 'lightgbm_phishing_model.txt'")

print("\n" + "="*80)
print("ANALYSE TERMINÉE")
print("="*80)

from sklearn.metrics import (
    classification_report, roc_curve, auc, accuracy_score,
    precision_recall_curve, confusion_matrix, average_precision_score,
    f1_score, precision_score,recall_score
)
import pickle

# Calculate final metrics
final_metrics = {
    'confusion_matrix': confusion_matrix(y_test, y_pred),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'fpr': fpr,
    'tpr': tpr,
    'roc_auc': roc_auc,
    'precision_curve': precision,
    'recall_curve': recall,
    'pr_auc': pr_auc,
    'feature_importance': lgbm_model.feature_importance(),
    'feature_names': selected_feature_names
}

# Save model, preprocessors, and metrics
model_data = {
    'model': lgbm_model,
    'power_transformer': power_transformer,
    'scaler': scaler,
    'selected_features_mask': selected_features_mask,
    'selected_feature_names': selected_feature_names,
    'feature_names': feature_names,
    'metrics': final_metrics
}
with open('lightgbm_phishing_model.pkl', 'wb') as file:
    pickle.dump(model_data, file)


Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import pickle
with open('lightgbm_phishing_model.pkl', 'wb') as file:
    pickle.dump(lgbm_model, file)

In [ ]:
# Add this at the end of your training script
print("\nSauvegarde du modèle et des informations de prétraitement...")
import pickle

# Save the model, feature selector, and other preprocessing components in one file
model_data = {
    'model': lgbm_model,
    'selected_features_mask': selected_features_mask,
    'selected_feature_names': selected_feature_names,
    'feature_names': feature_names
}

with open('lightgbm_phishing_model.pkl', 'wb') as file:
    pickle.dump(model_data, file)

print("Modèle et informations de prétraitement sauvegardés sous 'lightgbm_phishing_model.pkl'")


Sauvegarde du modèle et des informations de prétraitement...
Modèle et informations de prétraitement sauvegardés sous 'lightgbm_phishing_model.pkl'


In [ ]:
with open('lightgbm_phishing_model.pkl', 'rb') as file:
    model_data = pickle.load(file)
print(model_data.keys())


dict_keys(['model', 'selected_features_mask', 'selected_feature_names', 'feature_names'])


In [ ]:
from sklearn.metrics import (
    classification_report, roc_curve, auc, accuracy_score,
    precision_recall_curve, confusion_matrix, average_precision_score,
    f1_score, precision_score,recall_score
)
import pickle

# Calculate final metrics
final_metrics = {
    'confusion_matrix': confusion_matrix(y_test, y_pred),
    'accuracy': accuracy_score(y_test, y_pred),
    'precision': precision_score(y_test, y_pred),
    'recall': recall_score(y_test, y_pred),
    'f1': f1_score(y_test, y_pred),
    'fpr': fpr,
    'tpr': tpr,
    'roc_auc': roc_auc,
    'precision_curve': precision,
    'recall_curve': recall,
    'pr_auc': pr_auc,
    'feature_importance': lgbm_model.feature_importance(),
    'feature_names': selected_feature_names
}

# Save model, preprocessors, and metrics
model_data = {
    'model': lgbm_model,
    'power_transformer': power_transformer,
    'scaler': scaler,
    'selected_features_mask': selected_features_mask,
    'selected_feature_names': selected_feature_names,
    'feature_names': feature_names,
    'metrics': final_metrics
}
with open('lightgbm_phishing_model.pkl', 'wb') as file:
    pickle.dump(model_data, file)

In [ ]:
with open('lightgbm_phishing_model.pkl', 'rb') as file:
    model_data = pickle.load(file)
print(model_data.keys())

dict_keys(['model', 'power_transformer', 'scaler', 'selected_features_mask', 'selected_feature_names', 'feature_names', 'metrics'])
